# 🔁 LangGraph ETL Pipeline — Extract → Transform → Load
### Cell-by-cell walkthrough with Pandas for data operations

---
**What this notebook does:**  
Implements a strictly linear 3-step data cleaning pipeline orchestrated by **LangGraph**.  
Each node (Extract / Transform / Load) is an isolated Python function.  
A single typed `ETLState` dict flows through every node — no globals, no side-channels.

> ⚡ LangGraph is used **only for orchestration** — wiring nodes into a DAG and managing state handoff.  
> Pandas does all the actual data work.


## 📦 Cell 1 — Install Dependencies

In [1]:
# Install required libraries
# - langgraph  : DAG orchestration (the only orchestration layer used)
# - pandas     : all data operations
# - pyarrow    : optional, needed if writing .parquet output
!pip install langgraph pandas pyarrow -q


## 📥 Cell 2 — Imports

In [2]:
import json
import os
import re
import warnings
from typing import Any

import pandas as pd

# ── LangGraph imports ──────────────────────────────────────────────────────
# StateGraph : the main class to build a node graph
# END        : sentinel node — tells LangGraph the pipeline is finished
from langgraph.graph import StateGraph, END
# TypedDict  : used to define our typed state schema (ETLState)
from typing import TypedDict

warnings.filterwarnings("ignore")
print("✅ All imports successful")


✅ All imports successful


## 🗂️ Cell 3 — Define the State Object (`ETLState`)

This is the **single source of truth** passed between every LangGraph node.  
Think of it as a shared blackboard — each node reads from it and writes back to it.

| Field | Purpose |
|---|---|
| `source_path` | Input CSV file path |
| `dest_path` | Output cleaned file path |
| `metrics_path` | Where to write metrics.json |
| `raw_df` | DataFrame loaded in Extract |
| `cleaned_df` | DataFrame produced by Transform |
| `metrics` | Dict of stats (row counts, dups, etc.) |
| `errors` | List of QC failures |
| `warnings` | List of soft warnings |


In [3]:
# ETLState is a TypedDict — gives us type hints + dict interface
# LangGraph requires the state to be a dict or TypedDict
class ETLState(TypedDict):
    source_path: str      # e.g. "data/raw/input.csv"
    dest_path:   str      # e.g. "data/clean/output.csv"
    metrics_path: str     # e.g. "data/clean/metrics.json"
    raw_df:      Any      # populated by Extract node
    cleaned_df:  Any      # populated by Transform node
    metrics:     dict     # accumulated across all nodes
    errors:      list     # QC failures (non-fatal — pipeline still completes)
    warnings:    list     # soft issues (nulls filled, rows dropped, etc.)

print("✅ ETLState defined")


✅ ETLState defined


## 🔧 Cell 4 — Helper Utilities

In [4]:
def to_snake_case(col: str) -> str:
    """Normalize a column name: strip, lowercase, spaces→underscore, remove special chars."""
    col = col.strip().lower()
    col = re.sub(r"[\s\-]+", "_", col)   # spaces/hyphens → underscore
    col = re.sub(r"[^\w]", "", col)       # remove anything non-word
    return col

def log(step: str, msg: str):
    """Consistent log format: [STEP] message."""
    print(f"[{step}] {msg}")

print("✅ Helpers ready")


✅ Helpers ready


## 🟢 Cell 5 — Node 1: `extract`

**Responsibility:** Read the CSV from disk → populate `state["raw_df"]`.

LangGraph calls this function with the current state dict and expects a (possibly modified) state dict back.  
Defensive checks: file-not-found, empty file, malformed CSV.


In [5]:
# ── LangGraph NODE 1 ──────────────────────────────────────────────────────
# A LangGraph node is just a plain Python function:
#   Input  : ETLState dict
#   Output : ETLState dict (updated)
# LangGraph will call this automatically when the graph runs.

def extract(state: ETLState) -> ETLState:
    path = state["source_path"]
    log("EXTRACT", f"Reading → {path}")

    # Defensive: file must exist
    if not os.path.exists(path):
        raise FileNotFoundError(f"Source not found: {path}")

    try:
        df = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        raise ValueError(f"CSV is empty: {path}")
    except pd.errors.ParserError as e:
        raise ValueError(f"Malformed CSV: {e}")

    if df.empty:
        raise ValueError("CSV parsed but has 0 rows.")

    log("EXTRACT", f"✅ Loaded {len(df)} rows × {len(df.columns)} cols")

    # Write results back into state
    state["raw_df"]              = df
    state["metrics"]["raw_rows"] = len(df)
    state["metrics"]["raw_cols"] = len(df.columns)
    return state   # LangGraph passes this to the next node (transform)

print("✅ extract() node defined")


✅ extract() node defined


## 🟡 Cell 6 — Node 2: `transform`

**Responsibility:** Clean `raw_df` → produce `cleaned_df`.

Steps performed (in order):
1. Column name normalization (snake_case)
2. Duplicate removal (all-column basis)
3. Type coercion — numeric columns
4. Type coercion — date columns (`_date` / `_at` suffix)
5. Missing value handling (drop >50% null rows; fill numeric nulls with median)
6. **3 Data Quality checks** — failures go to `state["errors"]`


In [6]:
# ── LangGraph NODE 2 ──────────────────────────────────────────────────────
# Receives state from extract(), works on raw_df, writes cleaned_df back.

def transform(state: ETLState) -> ETLState:
    df     = state["raw_df"].copy()   # always copy — preserve raw in state
    warns  = state["warnings"]
    errors = state["errors"]
    log("TRANSFORM", "Starting ...")

    # ── Step 1: Column normalization ────────────────────────────────────────
    # Trim whitespace, lowercase, snake_case all column headers
    df.columns = [to_snake_case(c) for c in df.columns]
    log("TRANSFORM", f"Columns → {list(df.columns)}")

    # ── Step 2: Duplicate removal ───────────────────────────────────────────
    # Criteria: exact match across ALL columns (conservative default)
    dups = df.duplicated().sum()
    df   = df.drop_duplicates()
    state["metrics"]["duplicates_removed"] = int(dups)
    log("TRANSFORM", f"Duplicates removed: {dups}")

    # ── Step 3: Numeric type coercion ───────────────────────────────────────
    # For object cols: if >50% of values parse as float → treat as numeric
    for col in df.columns:
        if df[col].dtype == object:
            coerced = pd.to_numeric(df[col].astype(str).str.strip(), errors="coerce")
            if coerced.notna().sum() > 0.5 * len(df):
                df[col] = coerced
                log("TRANSFORM", f"  '{col}' → numeric")

    # ── Step 4: Date type coercion ──────────────────────────────────────────
    # Columns ending with '_date' or '_at' are parsed as datetime
    for col in df.columns:
        if ("_date" in col or col.endswith("_at")) and df[col].dtype == object:
            df[col] = pd.to_datetime(df[col], errors="coerce")
            log("TRANSFORM", f"  '{col}' → datetime")

    # ── Step 5: Missing value handling ─────────────────────────────────────
    # Assumption: drop rows where MORE than 50% of columns are null
    threshold = 0.5
    min_valid = int((1 - threshold) * len(df.columns)) + 1
    before    = len(df)
    df        = df.dropna(thresh=min_valid)
    dropped   = before - len(df)
    state["metrics"]["rows_dropped_null"] = dropped
    if dropped:
        warns.append(f"Dropped {dropped} rows (>50% null)")

    # Fill remaining numeric nulls with column median
    for col in df.select_dtypes(include="number").columns:
        nulls = df[col].isna().sum()
        if nulls:
            med = df[col].median()
            df[col] = df[col].fillna(med)
            warns.append(f"Filled {nulls} nulls in '{col}' with median={med:.2f}")

    # ── Step 6: Data Quality Checks ─────────────────────────────────────────

    # QC 1 — No fully-empty rows should remain
    full_null_rows = df.isnull().all(axis=1).sum()
    if full_null_rows:
        errors.append(f"QC1 FAIL: {full_null_rows} fully-empty rows remain")

    # QC 2 — age/count/qty columns must be non-negative
    for col in df.select_dtypes(include="number").columns:
        if any(k in col for k in ("age", "count", "qty", "quantity")):
            neg = (df[col] < 0).sum()
            if neg:
                errors.append(f"QC2 FAIL: '{col}' has {neg} negative values")

    # QC 3 — 'id' column (if present) must be unique
    if "id" in df.columns:
        dup_ids = df["id"].duplicated().sum()
        if dup_ids:
            errors.append(f"QC3 FAIL: 'id' has {dup_ids} duplicate values")

    # ── Accumulate metrics ──────────────────────────────────────────────────
    state["metrics"]["clean_rows"]  = len(df)
    state["metrics"]["null_counts"] = df.isnull().sum().to_dict()
    state["metrics"]["schema"]      = {c: str(df[c].dtype) for c in df.columns}

    if errors:
        log("TRANSFORM", f"⚠ QC errors: {errors}")
    log("TRANSFORM", f"✅ Done — {len(df)} rows remain")

    state["cleaned_df"] = df
    state["warnings"]   = warns
    state["errors"]     = errors
    return state   # LangGraph passes this to the load node

print("✅ transform() node defined")


✅ transform() node defined


## 🔵 Cell 7 — Node 3: `load`

**Responsibility:** Write `cleaned_df` to disk and emit `metrics.json`.  
Supports `.csv` or `.parquet` output (auto-detected from `dest_path` extension).  
Prints a final pipeline summary.


In [7]:
# ── LangGraph NODE 3 ──────────────────────────────────────────────────────
# Final node — no further handoff after this (connects to END).

def load(state: ETLState) -> ETLState:
    df    = state["cleaned_df"]
    dest  = state["dest_path"]
    mpath = state["metrics_path"]
    log("LOAD", f"Writing → {dest}")

    # Create output dirs if they don't exist
    os.makedirs(os.path.dirname(dest)  or ".", exist_ok=True)
    os.makedirs(os.path.dirname(mpath) or ".", exist_ok=True)

    # Write cleaned data (format inferred from extension)
    if dest.endswith(".parquet"):
        df.to_parquet(dest, index=False)
    else:
        df.to_csv(dest, index=False)

    # Write metrics artifact — converts all values to JSON-safe strings
    metrics_out = {
        **state["metrics"],
        "warnings": state["warnings"],
        "errors":   state["errors"],
    }
    with open(mpath, "w") as f:
        json.dump(metrics_out, f, indent=2, default=str)

    log("LOAD", f"Metrics → {mpath}")

    # ── Final summary ───────────────────────────────────────────────────────
    print("\n── PIPELINE SUMMARY ─────────────────────────────")
    for k, v in state["metrics"].items():
        if k not in ("null_counts", "schema"):
            print(f"  {k:30s}: {v}")
    print(f"  {'warnings':30s}: {len(state['warnings'])}")
    print(f"  {'QC errors':30s}: {len(state['errors'])}")
    print("─────────────────────────────────────────────────\n")

    log("LOAD", "✅ Pipeline complete")
    return state   # returned to LangGraph → hits END

print("✅ load() node defined")


✅ load() node defined


## 🔗 Cell 8 — Build the LangGraph DAG

This is where **LangGraph** does its job:
- `StateGraph(ETLState)` — declares a graph whose nodes share `ETLState`
- `.add_node(name, fn)` — registers each pipeline step
- `.set_entry_point("extract")` — first node to run
- `.add_edge(a, b)` — wires nodes linearly: extract → transform → load → END
- `.compile()` — validates the graph and returns a runnable object


In [8]:
# ── LangGraph: Build & compile the pipeline graph ─────────────────────────

def build_pipeline():
    # Create a new state graph typed to ETLState
    # LangGraph will validate that every node receives/returns this shape
    graph = StateGraph(ETLState)

    # Register the 3 nodes (name → function)
    graph.add_node("extract",   extract)
    graph.add_node("transform", transform)
    graph.add_node("load",      load)

    # Declare the entry point (first node LangGraph will call)
    graph.set_entry_point("extract")

    # Wire edges: strictly linear, no branching
    # LangGraph enforces this DAG — it won't call load before transform finishes
    graph.add_edge("extract",   "transform")
    graph.add_edge("transform", "load")
    graph.add_edge("load",      END)         # END = LangGraph sentinel for "done"

    # Compile validates the graph structure and returns an executable
    return graph.compile()

pipeline = build_pipeline()
print("✅ LangGraph pipeline compiled")
print("   Graph shape: extract → transform → load → END")


✅ LangGraph pipeline compiled
   Graph shape: extract → transform → load → END


## 🧪 Cell 9 — Generate Test Datasets

Three CSVs to exercise different scenarios:
| File | Scenario |
|---|---|
| `employees.csv` | Clean baseline — minimal issues |
| `orders.csv` | Dirty data — duplicates, nulls, negatives, bad types |
| `edge_case.csv` | Minimal rows — mostly null, tests drop threshold |


In [9]:
import numpy as np

os.makedirs("data/raw",   exist_ok=True)
os.makedirs("data/clean", exist_ok=True)

# ── Dataset 1: employees (clean baseline) ──────────────────────────────────
pd.DataFrame({
    "ID":         range(1, 101),
    "Full Name":  [f"User {i}" for i in range(1, 101)],
    "Age":        np.random.randint(22, 60, 100),
    "Salary":     np.random.randint(30000, 120000, 100),
    "Join Date":  pd.date_range("2018-01-01", periods=100, freq="W").astype(str),
    "Department": np.random.choice(["Eng", "HR", "Sales", "Finance"], 100),
}).to_csv("data/raw/employees.csv", index=False)
print("✅ data/raw/employees.csv  — 100 rows, clean")

# ── Dataset 2: orders (dirty) ──────────────────────────────────────────────
rows = 80
pd.DataFrame({
    "Order ID":           list(range(1, 71)) + [10,20,30,40,50,60,70,80,90,100],  # 10 dups
    "  Customer Name  ":  [f"  Cust {i}  " for i in range(rows)],   # whitespace in header+values
    "Amount ($)":         [str(np.random.randint(100,5000)) if i%7!=0 else "N/A" for i in range(rows)],
    "Count":              [np.random.randint(-5, 50) for _ in range(rows)],   # negatives → QC2 fail
    "Order Date":         pd.date_range("2023-01-01", periods=rows, freq="D").astype(str),
    "Status":             np.random.choice(["shipped","pending", None], rows),
}).to_csv("data/raw/orders.csv", index=False)
print("✅ data/raw/orders.csv     — 80 rows, dirty (dups, nulls, negatives)")

# ── Dataset 3: edge_case (mostly null) ─────────────────────────────────────
pd.DataFrame({
    "id":    [1, 2, 3, None, None],
    "value": [10.5, None, 33.0, None, None],
    "label": ["a", "b", None, None, None],
}).to_csv("data/raw/edge_case.csv", index=False)
print("✅ data/raw/edge_case.csv  — 5 rows, mostly null")


✅ data/raw/employees.csv  — 100 rows, clean
✅ data/raw/orders.csv     — 80 rows, dirty (dups, nulls, negatives)
✅ data/raw/edge_case.csv  — 5 rows, mostly null


## ▶️ Cell 10 — Run Pipeline on `employees.csv` (clean baseline)

In [10]:
initial_state: ETLState = {
    "source_path":  "data/raw/employees.csv",
    "dest_path":    "data/clean/employees_clean.csv",
    "metrics_path": "data/clean/employees_metrics.json",
    "raw_df":       None,
    "cleaned_df":   None,
    "metrics":      {},   # LangGraph threads this dict through all 3 nodes
    "errors":       [],
    "warnings":     [],
}

# ── LangGraph: invoke() kicks off the graph from the entry point ───────────
# It calls extract() → transform() → load() in order, passing state between them
result = pipeline.invoke(initial_state)


[EXTRACT] Reading → data/raw/employees.csv
[EXTRACT] ✅ Loaded 100 rows × 6 cols
[TRANSFORM] Starting ...
[TRANSFORM] Columns → ['id', 'full_name', 'age', 'salary', 'join_date', 'department']
[TRANSFORM] Duplicates removed: 0
[TRANSFORM]   'join_date' → datetime
[TRANSFORM] ✅ Done — 100 rows remain
[LOAD] Writing → data/clean/employees_clean.csv
[LOAD] Metrics → data/clean/employees_metrics.json

── PIPELINE SUMMARY ─────────────────────────────
  raw_rows                      : 100
  raw_cols                      : 6
  duplicates_removed            : 0
  rows_dropped_null             : 0
  clean_rows                    : 100
  warnings                      : 0
  QC errors                     : 0
─────────────────────────────────────────────────

[LOAD] ✅ Pipeline complete


## ▶️ Cell 11 — Run Pipeline on `orders.csv` (dirty data)

In [11]:
result_orders = pipeline.invoke({
    "source_path":  "data/raw/orders.csv",
    "dest_path":    "data/clean/orders_clean.csv",
    "metrics_path": "data/clean/orders_metrics.json",
    "raw_df":       None,
    "cleaned_df":   None,
    "metrics":      {},
    "errors":       [],
    "warnings":     [],
})


[EXTRACT] Reading → data/raw/orders.csv
[EXTRACT] ✅ Loaded 80 rows × 6 cols
[TRANSFORM] Starting ...
[TRANSFORM] Columns → ['order_id', 'customer_name', 'amount_', 'count', 'order_date', 'status']
[TRANSFORM] Duplicates removed: 0
[TRANSFORM]   'order_date' → datetime
[TRANSFORM] ⚠ QC errors: ["QC2 FAIL: 'count' has 7 negative values"]
[TRANSFORM] ✅ Done — 80 rows remain
[LOAD] Writing → data/clean/orders_clean.csv
[LOAD] Metrics → data/clean/orders_metrics.json

── PIPELINE SUMMARY ─────────────────────────────
  raw_rows                      : 80
  raw_cols                      : 6
  duplicates_removed            : 0
  rows_dropped_null             : 0
  clean_rows                    : 80
  warnings                      : 1
  QC errors                     : 1
─────────────────────────────────────────────────

[LOAD] ✅ Pipeline complete


## ▶️ Cell 12 — Run Pipeline on `edge_case.csv` (minimal rows)

In [12]:
result_edge = pipeline.invoke({
    "source_path":  "data/raw/edge_case.csv",
    "dest_path":    "data/clean/edge_case_clean.csv",
    "metrics_path": "data/clean/edge_case_metrics.json",
    "raw_df":       None,
    "cleaned_df":   None,
    "metrics":      {},
    "errors":       [],
    "warnings":     [],
})


[EXTRACT] Reading → data/raw/edge_case.csv
[EXTRACT] ✅ Loaded 5 rows × 3 cols
[TRANSFORM] Starting ...
[TRANSFORM] Columns → ['id', 'value', 'label']
[TRANSFORM] Duplicates removed: 1
[TRANSFORM] ✅ Done — 3 rows remain
[LOAD] Writing → data/clean/edge_case_clean.csv
[LOAD] Metrics → data/clean/edge_case_metrics.json

── PIPELINE SUMMARY ─────────────────────────────
  raw_rows                      : 5
  raw_cols                      : 3
  duplicates_removed            : 1
  rows_dropped_null             : 1
  clean_rows                    : 3
  warnings                      : 2
  QC errors                     : 0
─────────────────────────────────────────────────

[LOAD] ✅ Pipeline complete


## 🔍 Cell 13 — Inspect Cleaned Output & Metrics

In [13]:
# Preview cleaned employees DataFrame
print("── Cleaned Employees (first 5 rows) ──")
display(result["cleaned_df"].head())

print("\n── Cleaned Orders (first 5 rows) ──")
display(result_orders["cleaned_df"].head())

print("\n── Cleaned Edge Case ──")
display(result_edge["cleaned_df"])


── Cleaned Employees (first 5 rows) ──


,id,full_name,age,salary,join_date,department
0,1,User 1,59,100854,2018-01-07,Finance
1,2,User 2,53,65283,2018-01-14,Eng
2,3,User 3,53,51589,2018-01-21,Eng
3,4,User 4,56,78705,2018-01-28,Sales
4,5,User 5,37,111584,2018-02-04,Eng



── Cleaned Orders (first 5 rows) ──


,order_id,customer_name,amount_,count,order_date,status
0,1,Cust 0,2339.0,3,2023-01-01,NaN
1,2,Cust 1,2685.0,4,2023-01-02,NaN
2,3,Cust 2,518.0,9,2023-01-03,shipped
3,4,Cust 3,1495.0,17,2023-01-04,pending
4,5,Cust 4,862.0,25,2023-01-05,pending



── Cleaned Edge Case ──


,id,value,label
0,1.0,10.50,a
1,2.0,21.75,b
2,3.0,33.00,NaN


## 📊 Cell 14 — View Metrics JSON

In [14]:
with open("data/clean/orders_metrics.json") as f:
    metrics = json.load(f)

print(json.dumps(metrics, indent=2, default=str))


{
  "raw_rows": 80,
  "raw_cols": 6,
  "duplicates_removed": 0,
  "rows_dropped_null": 0,
  "clean_rows": 80,
  "null_counts": {
    "order_id": 0,
    "customer_name": 0,
    "amount_": 0,
    "count": 0,
    "order_date": 0,
    "status": 33
  },
  "schema": {
    "order_id": "int64",
    "customer_name": "object",
    "amount_": "float64",
    "count": "int64",
    "order_date": "datetime64[ns]",
    "status": "object"
  },
  "warnings": [
    "Filled 12 nulls in 'amount_' with median=2339.00"
  ],
  "errors": [
    "QC2 FAIL: 'count' has 7 negative values"
  ]
}


## 📌 Cell 15 — Where LangGraph Is Used (Summary)

| Location | LangGraph API | Purpose |
|---|---|---|
| Cell 2 | `from langgraph.graph import StateGraph, END` | Import the graph builder and terminal sentinel |
| Cell 3 | `class ETLState(TypedDict)` | Define the state schema LangGraph threads through nodes |
| Cell 8 | `StateGraph(ETLState)` | Create a typed graph instance |
| Cell 8 | `.add_node("extract", extract)` | Register each pipeline step as a named node |
| Cell 8 | `.set_entry_point("extract")` | Declare the first node LangGraph will call |
| Cell 8 | `.add_edge(a, b)` | Wire nodes linearly — enforces execution order |
| Cell 8 | `.add_edge("load", END)` | Tell LangGraph the pipeline terminates after load |
| Cell 8 | `.compile()` | Validate graph structure, return executable |
| Cells 10–12 | `pipeline.invoke(state)` | Run the full graph with an initial state dict |

**Pandas does:** `read_csv`, `drop_duplicates`, `to_numeric`, `to_datetime`, `fillna`, `dropna`, `to_csv`  
**LangGraph does:** node registration, edge wiring, state handoff, execution ordering


In [ ]:
result_products = pipeline.invoke({
    "source_path":  "Datasets/products.csv",
    "dest_path":    "data/clean/products_clean.csv",
    "metrics_path": "data/clean/products_metrics.json",
    "raw_df":       None,
    "cleaned_df":   None,
    "metrics":      {},
    "errors":       [],
    "warnings":     [],
})

result_transactions = pipeline.invoke({
    "source_path":  "Datasets/transactions.csv",
    "dest_path":    "data/clean/transactions_clean.csv",
    "metrics_path": "data/clean/transactions_metrics.json",
    "raw_df":       None,
    "cleaned_df":   None,
    "metrics":      {},
    "errors":       [],
    "warnings":     [],
})

result_users = pipeline.invoke({
    "source_path":  "Datasets/users.csv",
    "dest_path":    "data/clean/users_clean.csv",
    "metrics_path": "data/clean/users_metrics.json",
    "raw_df":       None,
    "cleaned_df":   None,
    "metrics":      {},
    "errors":       [],
    "warnings":     [],
})

In [ ]:
print("── Cleaned Products (first 5 rows) ──")
display(result_products["cleaned_df"].head())

print("\n── Cleaned Transactions (first 5 rows) ──")
display(result_transactions["cleaned_df"].head())

print("\n── Cleaned Users (first 5 rows) ──")
display(result_users["cleaned_df"].head())